In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install tf_keras -q

In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from datasets import load_dataset

In [3]:
dataset = load_dataset("ucirvine/sms_spam")
texts = np.array(dataset["train"]["sms"])
labels = np.array(dataset["train"]["label"])
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

In [4]:
input_layer = Input(shape=(), dtype=tf.string, name="text_input")
preprocessor_layer = hub.KerasLayer("https://www.kaggle.com/models/tensorflow/bert/TensorFlow2/en-uncased-preprocess/3", name="preprocessing")
encoder_layer = hub.KerasLayer("https://www.kaggle.com/models/tensorflow/bert/TensorFlow2/bert-en-uncased-l-12-h-128-a-2/2", trainable=True, name="BERT_encoder")

In [5]:
x = preprocessor_layer(input_layer)
x = encoder_layer(x)["pooled_output"]
x = Dropout(0.1, name="dropout")(x)
output_layer = Dense(2, activation="softmax", name="output")(x)  # spam or not spam
model = Model(inputs=input_layer, outputs=output_layer)

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [7]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)

Epoch 1/5
140/140 [==============================] - 503s 3s/step - loss: 0.1258 - accuracy: 0.9578 - val_loss: 0.0484 - val_accuracy: 0.9830
Epoch 2/5
140/140 [==============================] - 487s 3s/step - loss: 0.0463 - accuracy: 0.9872 - val_loss: 0.0438 - val_accuracy: 0.9848
Epoch 3/5
140/140 [==============================] - 460s 3s/step - loss: 0.0347 - accuracy: 0.9908 - val_loss: 0.0393 - val_accuracy: 0.9910
Epoch 4/5
140/140 [==============================] - 487s 3s/step - loss: 0.0253 - accuracy: 0.9933 - val_loss: 0.0458 - val_accuracy: 0.9839
Epoch 5/5
140/140 [==============================] - 468s 3s/step - loss: 0.0173 - accuracy: 0.9960 - val_loss: 0.0388 - val_accuracy: 0.9892
